# XAI-ED — Notebook 1: Data Exploration & Equity Analysis

This notebook provides a thorough exploratory analysis of the synthetic student dataset,
including:
- Feature distributions and class balance
- Correlation heatmap
- Demographic equity gaps (mastery rate by SES, first-gen, gender)
- Label calibration verification

**Dataset:** `data/student_data.csv` (3,000 students, 10 academic features + 3 demographic features)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from src.config import FEATURE_COLUMNS, TARGET_COLUMN
from src.data_gen import generate_student_dataset

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='darkgrid', palette='deep')
print('Libraries loaded.')

In [ ]:
# Load or generate dataset
import os
CSV_PATH = '../data/student_data.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded existing dataset: {len(df)} rows')
else:
    df = generate_student_dataset(n=3000, seed=42)
    print(f'Generated dataset: {len(df)} rows')

df.head()

## 1. Dataset Summary

In [ ]:
print('Shape:', df.shape)
print(f'\nMastery rate (overall): {df[TARGET_COLUMN].mean():.3f}')
print(f'At-risk rate:           {1 - df[TARGET_COLUMN].mean():.3f}')
print(f'\nClass counts:')
print(df[TARGET_COLUMN].value_counts().rename({0: 'At-Risk', 1: 'On-Track'}))
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.describe().round(3)

## 2. Feature Distributions

In [ ]:
FRIENDLY = {
    'study_time_min': 'Study Time (min/wk)',
    'practice_completion_rate': 'Practice Completion',
    'avg_quiz_score': 'Avg Quiz Score',
    'quiz_attempts': 'Quiz Attempts',
    'hint_usage_rate': 'Hint Usage Rate',
    'attendance_rate': 'Attendance Rate',
    'days_since_last_activity': 'Days Since Last Activity',
    'stress_index': 'Stress Index',
    'prereq_mastery': 'Prereq Mastery',
    'device_reliability': 'Device Reliability',
}

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLUMNS):
    ax = axes[i]
    at_risk = df[df[TARGET_COLUMN] == 0][feat]
    on_track = df[df[TARGET_COLUMN] == 1][feat]
    ax.hist(at_risk,  bins=30, alpha=0.6, color='#ef5350', label='At-Risk',  density=True)
    ax.hist(on_track, bins=30, alpha=0.6, color='#42a5f5', label='On-Track', density=True)
    ax.set_title(FRIENDLY.get(feat, feat), fontsize=10)
    ax.set_ylabel('Density')
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Mastery Label', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/feature_distributions.png')

## 3. Correlation Heatmap

In [ ]:
corr_cols = FEATURE_COLUMNS + [TARGET_COLUMN]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', center=0,
    cmap='RdBu_r', linewidths=0.5, ax=ax,
    xticklabels=[FRIENDLY.get(c, c) for c in corr_cols],
    yticklabels=[FRIENDLY.get(c, c) for c in corr_cols],
)
ax.set_title('Feature Correlation Matrix (including mastery target)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop features correlated with mastery:')
print(corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False).round(3))

## 4. Equity Gap Analysis

A key research contribution is the realistic simulation of demographic equity gaps.
This section quantifies those gaps.

In [ ]:
# SES equity gap
df['ses_group'] = pd.qcut(df['ses_index'], q=3, labels=['Low SES', 'Medium SES', 'High SES'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# SES mastery rate
ses_mastery = df.groupby('ses_group')[TARGET_COLUMN].mean()
axes[0].bar(ses_mastery.index, ses_mastery.values, color=['#ef5350', '#ffa726', '#42a5f5'], edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Mastery Rate')
axes[0].set_title('Mastery Rate by SES Tier', fontweight='bold')
for i, v in enumerate(ses_mastery.values):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# First-gen mastery rate
fg_mastery = df.groupby('first_gen')[TARGET_COLUMN].mean()
fg_labels  = ['Continuing', 'First-Gen']
axes[1].bar(fg_labels, fg_mastery.values, color=['#42a5f5', '#ef5350'], edgecolor='white')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Mastery Rate')
axes[1].set_title('Mastery Rate: First-Gen vs Continuing', fontweight='bold')
for i, v in enumerate(fg_mastery.values):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Gender mastery rate
gender_mastery = df.groupby('gender')[TARGET_COLUMN].mean()
gender_labels  = {0: 'Female', 1: 'Male', 2: 'Non-binary'}
g_labels = [gender_labels[g] for g in gender_mastery.index]
axes[2].bar(g_labels, gender_mastery.values, color=['#ab47bc', '#42a5f5', '#66bb6a'], edgecolor='white')
axes[2].set_ylim(0, 1)
axes[2].set_ylabel('Mastery Rate')
axes[2].set_title('Mastery Rate by Gender', fontweight='bold')
for i, v in enumerate(gender_mastery.values):
    axes[2].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Equity Gaps in Student Mastery Rates', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/equity_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table of equity gaps
gaps = pd.DataFrame({
    'Axis': ['SES', 'SES', 'SES', 'First-Gen', 'First-Gen', 'Gender', 'Gender', 'Gender'],
    'Group': ['Low', 'Medium', 'High', 'First-Gen', 'Continuing', 'Female', 'Male', 'Non-binary'],
    'N': [
        len(df[df['ses_group'] == 'Low SES']),
        len(df[df['ses_group'] == 'Medium SES']),
        len(df[df['ses_group'] == 'High SES']),
        int((df['first_gen'] == 1).sum()),
        int((df['first_gen'] == 0).sum()),
        int((df['gender'] == 0).sum()),
        int((df['gender'] == 1).sum()),
        int((df['gender'] == 2).sum()),
    ],
    'Mastery Rate': [
        df[df['ses_group'] == 'Low SES'][TARGET_COLUMN].mean(),
        df[df['ses_group'] == 'Medium SES'][TARGET_COLUMN].mean(),
        df[df['ses_group'] == 'High SES'][TARGET_COLUMN].mean(),
        df[df['first_gen'] == 1][TARGET_COLUMN].mean(),
        df[df['first_gen'] == 0][TARGET_COLUMN].mean(),
        df[df['gender'] == 0][TARGET_COLUMN].mean(),
        df[df['gender'] == 1][TARGET_COLUMN].mean(),
        df[df['gender'] == 2][TARGET_COLUMN].mean(),
    ],
}).round(3)
print(gaps.to_string(index=False))

## 5. Feature Differences by Demographic Group

Verifying that the equity adjustments applied in `data_gen.py` produce
realistic structural differences in academic features.

In [ ]:
features_to_compare = [
    'study_time_min', 'attendance_rate', 'device_reliability',
    'prereq_mastery', 'quiz_attempts', 'stress_index'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(features_to_compare):
    ax = axes[i]
    low_ses  = df[df['ses_group'] == 'Low SES'][feat]
    high_ses = df[df['ses_group'] == 'High SES'][feat]
    fg       = df[df['first_gen'] == 1][feat]
    cont     = df[df['first_gen'] == 0][feat]
    ax.boxplot(
        [low_ses, high_ses, fg, cont],
        labels=['Low SES', 'High SES', 'First-Gen', 'Continuing'],
        patch_artist=True,
        boxprops=dict(facecolor='#1e3a5f', color='white'),
        medianprops=dict(color='#42a5f5', linewidth=2),
        whiskerprops=dict(color='white'),
        capprops=dict(color='white'),
        flierprops=dict(marker='.', color='gray', alpha=0.3),
    )
    ax.set_title(FRIENDLY.get(feat, feat), fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Feature Distributions by Demographic Group', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/feature_by_demographic.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Label Calibration Verification

The mastery label is generated via a sigmoid of a linear combination of features.
We verify that the bias term correctly calibrates the base rate.

In [ ]:
print('Overall mastery rate:', df[TARGET_COLUMN].mean().round(4))
print('Target range: 0.78 – 0.88 (calibrated to ~0.82 with bias=-6.0)\n')

# Show how each feature correlates with mastery
correlations = df[FEATURE_COLUMNS + [TARGET_COLUMN]].corr()[TARGET_COLUMN].drop(TARGET_COLUMN)
correlations = correlations.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#42a5f5' if v > 0 else '#ef5350' for v in correlations.values]
ax.barh([FRIENDLY.get(f, f) for f in correlations.index], correlations.values, color=colors)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--')
ax.set_xlabel('Pearson Correlation with Mastery Label')
ax.set_title('Feature–Mastery Correlations', fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(ax.patches, correlations.values):
    x = bar.get_width()
    ax.text(x + 0.005 if x >= 0 else x - 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if x >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/feature_mastery_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
**Notebook complete.** Proceed to `02_model_comparison.ipynb` for training and evaluation.